# Lumut Home: Data Audit

**Purpose:** inspect the five raw files *before* changing anything, and quantify every data-quality issue.
Cleaning decisions are made in `01_data_cleaning.ipynb`.

**Method.** The same checks were run on every table:

| # | Check | What it catches |
| --- | --- | --- |
| 1 | Shape and types (`.shape`, `.info()`) | Wrong row counts, text where dates/numbers belong |
| 2 | Distinct values of text columns (`.value_counts()`) | Spelling and case variants |
| 3 | Blank counts (`.isna().sum()`) | Missing data |
| 4 | Extremes of number columns (`.describe()`) | Negatives, zeros, decimal-shift outliers |
| 5 | Business-logic tests | Impossible values (clicks > impressions, end date < start date, signup after data ends) |
| 6 | Key tests (`.duplicated()`, `.isin()`) | Duplicate rows, IDs missing from the parent table |
| 7 | Comparison with expectation | A missing month, mixed date formats |

Nothing in this notebook modifies the raw files.

## 1. Load and size the data

In [1]:
import os
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

DATA_DIR = "../data_raw/"       # original, untouched CSV files

campaigns = pd.read_csv(DATA_DIR + "campaigns.csv")
ads       = pd.read_csv(DATA_DIR + "ad_performance_daily.csv")
orders    = pd.read_csv(DATA_DIR + "orders.csv")
customers = pd.read_csv(DATA_DIR + "customers.csv")
fx        = pd.read_csv(DATA_DIR + "fx_rates_monthly.csv")

# (rows, columns) for each table
sizes = pd.DataFrame({
    "table": ["campaigns", "ad_performance_daily", "orders", "customers", "fx_rates_monthly"],
    "rows":    [len(campaigns), len(ads), len(orders), len(customers), len(fx)],
    "columns": [campaigns.shape[1], ads.shape[1], orders.shape[1], customers.shape[1], fx.shape[1]],
})
sizes

,table,rows,columns
0,campaigns,111,13
1,ad_performance_daily,73628,11
2,orders,47669,12
3,customers,30499,6
4,fx_rates_monthly,21,2


### Integrity check on the raw files
Raw files must never be opened and re-saved in a spreadsheet: doing so silently rewrites dates and changes file sizes.
This check compares each file's size with the size at extraction.

In [2]:
original_sizes = {"campaigns.csv": 16440, "ad_performance_daily.csv": 5507884,
                  "orders.csv": 3846389, "customers.csv": 1728537, "fx_rates_monthly.csv": 391}
for name, size in original_sizes.items():
    print(f"{name}: unchanged = {os.path.getsize(DATA_DIR + name) == size}")

campaigns.csv: unchanged = True
ad_performance_daily.csv: unchanged = True
orders.csv: unchanged = True
customers.csv: unchanged = True
fx_rates_monthly.csv: unchanged = True


## 2. Table-by-table inspection

### fx_rates_monthly

In [3]:
fx.info()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   month       21 non-null     str    
 1   gbp_to_usd  21 non-null     float64
dtypes: float64(1), str(1)
memory usage: 468.0 bytes


In [4]:
fx.head()

,month,gbp_to_usd
0,2025-01-01,1.2625
1,2025-02-01,1.2644
2,2025-03-01,1.2467
3,2025-04-01,1.2489
4,2025-05-01,1.2416


### campaigns

In [5]:
campaigns.info()

<class 'pandas.DataFrame'>
RangeIndex: 111 entries, 0 to 110
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   campaign_id     111 non-null    str    
 1   campaign_name   111 non-null    str    
 2   channel         111 non-null    str    
 3   platform        111 non-null    str    
 4   campaign_type   111 non-null    str    
 5   objective       111 non-null    str    
 6   market          111 non-null    str    
 7   start_date      111 non-null    str    
 8   end_date        111 non-null    str    
 9   planned_budget  105 non-null    float64
 10  currency        111 non-null    str    
 11  status          111 non-null    str    
 12  owner_team      111 non-null    str    
dtypes: float64(1), str(12)
memory usage: 11.4 KB


In [6]:
campaigns.isna().sum()

campaign_id       0
campaign_name     0
channel           0
platform          0
campaign_type     0
objective         0
market            0
start_date        0
end_date          0
planned_budget    6
currency          0
status            0
owner_team        0
dtype: int64

In [7]:
# Text columns: how many distinct spellings exist?
print(campaigns["channel"].value_counts(), "\n")
print(campaigns["platform"].value_counts())

channel
Paid Search        36
Paid Social        32
Shopping           11
Display             7
Video               6
paid social         3
Email               3
Google Shopping     2
email               2
Social              2
Search              2
paid search         1
YouTube Ads         1
display             1
GDN                 1
PPC                 1
Name: count, dtype: int64 

platform
Google Ads                41
Meta                      24
Microsoft Ads              9
Google Display Network     8
YouTube                    5
Pinterest                  4
Klaviyo                    3
Facebook Ads               3
Meta Ads                   2
klaviyo                    2
pinterest                  2
AdWords                    1
Youtube                    1
TikTok                     1
Google Display             1
google ads                 1
YouTube Ads                1
facebook                   1
Bing Ads                   1
Name: count, dtype: int64


### ad_performance_daily

In [8]:
ads.info()

<class 'pandas.DataFrame'>
RangeIndex: 73628 entries, 0 to 73627
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   date              73628 non-null  str    
 1   campaign_id       73628 non-null  str    
 2   ad_group_id       73628 non-null  str    
 3   ad_group_name     73628 non-null  str    
 4   device            72532 non-null  str    
 5   impressions       73628 non-null  int64  
 6   clicks            73628 non-null  int64  
 7   spend             73261 non-null  float64
 8   currency          73628 non-null  str    
 9   conversions       73628 non-null  float64
 10  conversion_value  73407 non-null  float64
dtypes: float64(3), int64(2), str(6)
memory usage: 6.2 MB


In [9]:
ads.isna().sum()

date                   0
campaign_id            0
ad_group_id            0
ad_group_name          0
device              1096
impressions            0
clicks                 0
spend                367
currency               0
conversions            0
conversion_value     221
dtype: int64

In [10]:
print(ads["device"].value_counts(dropna=False), "\n")
print(ads["currency"].value_counts())

device
Mobile     22584
Desktop    22527
Tablet     22523
NaN         1096
tablet       765
DESKTOP      733
mobile       729
TABLET       726
desktop      693
MOBILE       665
Unknown      587
Name: count, dtype: int64 

currency
USD    39169
GBP    31509
usd      552
$        549
US$      536
Gbp      444
gbp      435
£        434
Name: count, dtype: int64


In [11]:
# Summary statistics: compare min with 25%, max with 75%, mean with median
ads.describe()

,impressions,clicks,spend,conversions,conversion_value
count,73628.000000,73628.000000,73261.000000,73628.000000,73407.000000
mean,3356.199408,42.524977,23.429688,0.812667,60.457468
std,9612.343518,686.860803,95.725264,1.357308,104.593767
min,6.000000,0.000000,-143.880000,0.000000,0.000000
25%,297.000000,8.000000,4.220000,0.000000,0.000000
50%,958.000000,20.000000,11.870000,0.000000,0.000000
75%,3283.000000,44.000000,29.170000,1.090000,86.660000
max,229899.000000,176892.000000,10017.000000,16.420000,1553.960000


### orders

In [12]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 47669 entries, 0 to 47668
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   order_id                 47669 non-null  str    
 1   customer_id              47480 non-null  str    
 2   order_date               47669 non-null  str    
 3   country                  47669 non-null  str    
 4   currency                 47669 non-null  str    
 5   product_category         47669 non-null  str    
 6   order_value              47669 non-null  float64
 7   discount_amount          47669 non-null  float64
 8   shipping_cost            46237 non-null  float64
 9   is_refunded              47669 non-null  str    
 10  first_touch_campaign_id  43951 non-null  str    
 11  last_touch_campaign_id   39730 non-null  str    
dtypes: float64(3), str(9)
memory usage: 4.4 MB


In [13]:
orders.isna().sum()

order_id                      0
customer_id                 189
order_date                    0
country                       0
currency                      0
product_category              0
order_value                   0
discount_amount               0
shipping_cost              1432
is_refunded                   0
first_touch_campaign_id    3718
last_touch_campaign_id     7939
dtype: int64

In [14]:
print(orders["country"].value_counts(), "\n")
print(orders["is_refunded"].value_counts())

country
US                21864
UK                20086
us                 1015
United States       992
USA                 980
uk                  702
United Kingdom      688
U.K.                684
GB                  658
Name: count, dtype: int64 

is_refunded
N        33748
FALSE     3762
0         3740
No        3731
Y         2016
Yes        254
1          211
TRUE       207
Name: count, dtype: int64


In [15]:
orders[['order_value', 'discount_amount', 'shipping_cost']].describe()

,order_value,discount_amount,shipping_cost
count,47669.000000,47669.000000,46237.000000
mean,78.096290,2.291728,2.825976
std,59.317561,6.022539,2.579950
min,-205.790000,0.000000,0.000000
25%,41.280000,0.000000,0.000000
50%,61.160000,0.000000,3.950000
75%,94.510000,0.000000,5.990000
max,1727.640000,181.180000,5.990000


### customers

In [16]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 30499 entries, 0 to 30498
Data columns (total 6 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   customer_id              30499 non-null  str  
 1   signup_date              30499 non-null  str  
 2   country                  30499 non-null  str  
 3   email_hash               30347 non-null  str  
 4   marketing_opt_in         30499 non-null  str  
 5   crm_acquisition_channel  27755 non-null  str  
dtypes: str(6)
memory usage: 1.4 MB


In [17]:
customers.isna().sum()

customer_id                   0
signup_date                   0
country                       0
email_hash                  152
marketing_opt_in              0
crm_acquisition_channel    2744
dtype: int64

In [18]:
print(customers["country"].value_counts(), "\n")
print(customers["crm_acquisition_channel"].value_counts(dropna=False))

country
US                15018
UK                12432
USA                1728
United Kingdom     1321
Name: count, dtype: int64 

crm_acquisition_channel
Paid Search         10214
Shopping             7506
Paid Social          3802
Direct / Organic     3336
NaN                  2744
google                866
google shopping       658
Display               597
facebook              345
Email                 184
Video                 168
display                54
email                  13
youtube                12
Name: count, dtype: int64


## 3. Investigating the red flags in `ad_performance_daily`

`describe()` showed a negative minimum and a maximum far above the 75th percentile for `spend`, and very large `clicks`.
The cells below locate the rows behind those numbers.

In [19]:
# The 10 largest spend values. Look at the whole-number values and the huge implied cost per click.
top = ads.sort_values("spend", ascending=False).head(10).copy()
top["implied_cpc"] = (top["spend"] / top["clicks"]).round(2)
top[["date", "campaign_id", "clicks", "spend", "currency", "implied_cpc"]]

,date,campaign_id,clicks,spend,currency,implied_cpc
64887,2026-09-10,CMP-0090,154,10017.0,GBP,65.05
37705,19/11/2025,CMP-0049,77,9770.0,USD,126.88
45139,2025-12-20,CMP-0061,59,9023.0,GBP,152.93
37769,2025-11-27,CMP-0049,64,8730.0,USD,136.41
24477,2025-08-10,CMP-0022,42,7106.0,USD,169.19
5909,2026-02-09,CMP-0002,82,7033.0,GBP,85.77
5070,2025-09-08,CMP-0002,88,6212.0,GBP,70.59
38750,2025-11-23,CMP-0050,108,5434.0,USD,50.31
17975,2025-03-25,CMP-0011,39,3590.0,USD,92.05
13004,2025-12-25,CMP-0004,32,3565.0,USD,111.41


In [20]:
# Negative spend
ads[ads["spend"] < 0].shape[0], ads[ads["spend"] < 0].head()

(73,
             date campaign_id ad_group_id ad_group_name   device  impressions  clicks  spend currency  conversions  conversion_value
 583   2025-04-12    CMP-0001   AG-0001-2  Brand Phrase   Mobile          250      25  -4.82      GBP         3.72            196.01
 666   2025-04-27    CMP-0001   AG-0001-1   Brand Exact   Tablet           33       6  -1.48      GBP         0.00              0.00
 3459  2026-09-27    CMP-0001   AG-0001-1   Brand Exact  Desktop          143      11  -3.46      GBP         0.00              0.00
 3545  2026-10-12    CMP-0001   AG-0001-2  Brand Phrase   Mobile          224      20  -3.34      GBP         0.00              0.00
 6526  2026-06-06    CMP-0002   AG-0002-1   High Margin  Desktop         1723      29 -23.74      GBP         0.00              0.00)

In [21]:
# Clicks greater than impressions is impossible
bad_clicks = ads[ads["clicks"] > ads["impressions"]]
bad_clicks.shape[0], bad_clicks.head()

(148,
             date campaign_id ad_group_id ad_group_name   device  impressions  clicks  spend currency  conversions  conversion_value
 380   07/03/2025    CMP-0001   AG-0001-2  Brand Phrase   Tablet           28      54   0.55      GBP         0.00               0.0
 561   2025-04-08    CMP-0001   AG-0001-2  Brand Phrase   Tablet           35      53   0.64      GBP         0.00               0.0
 2131  2026-01-27    CMP-0001   AG-0001-1   Brand Exact   Tablet           44      64   0.37      GBP         0.00               0.0
 2699  2026-05-11    CMP-0001   AG-0001-2  Brand Phrase  Desktop          122     126   5.18      GBP         1.11              60.4
 3085  2026-07-18    CMP-0001   AG-0001-2  Brand Phrase   Mobile          237     243   6.08      GBP         0.00               0.0)

In [22]:
# Date formats: the file mixes YYYY-MM-DD with DD/MM/YYYY
iso = ads["date"].str.match(r"^\d{4}-\d{2}-\d{2}$")
print("ISO dates:", iso.sum(), "| other format:", (~iso).sum())
ads.loc[~iso, "date"].head()

ISO dates: 72155 | other format: 1473


34     06/01/2025
93     17/01/2025
146    26/01/2025
209    05/02/2025
269    15/02/2025
Name: date, dtype: str

In [23]:
# A date like 07/03/2025 is ambiguous (7 March or 3 July?). The file is in date order,
# so neighbouring rows settle it: compare the row's position with nearby ISO dates.
ads.loc[[378, 379, 380, 381, 382], ["date", "campaign_id"]]

,date,campaign_id
378,2025-03-07,CMP-0001
379,2025-03-07,CMP-0001
380,07/03/2025,CMP-0001
381,2025-03-08,CMP-0001
382,2025-03-08,CMP-0001


## 4. Keys and relationships

In [24]:
# campaign_id formatting problems (lowercase / spaces)
bad_format = ~ads["campaign_id"].str.match(r"^CMP-\d{4}$")
print("campaign_id with wrong format:", bad_format.sum())

# After normalising, do all IDs exist in campaigns?
normalised = ads["campaign_id"].str.strip().str.upper()
orphans = ~normalised.isin(campaigns["campaign_id"])
print("rows with campaign_id not in campaigns:", orphans.sum())
print(sorted(normalised[orphans].unique())[:5], "...")

# Exact duplicate rows
print("exact duplicate rows in ads:", ads.duplicated().sum())
print("duplicate order_id in orders:", orders["order_id"].duplicated().sum())

campaign_id with wrong format: 220


rows with campaign_id not in campaigns: 112
['CMP-9001', 'CMP-9002', 'CMP-9004', 'CMP-9005', 'CMP-9008'] ...


exact duplicate rows in ads: 584
duplicate order_id in orders: 284


In [25]:
# Campaigns whose end date is before the start date
bad_dates = campaigns[pd.to_datetime(campaigns["end_date"]) < pd.to_datetime(campaigns["start_date"])]
bad_dates[["campaign_id", "start_date", "end_date"]]

,campaign_id,start_date,end_date
11,CMP-0012,2025-07-14,2025-03-02
42,CMP-0043,2026-01-05,2025-09-25
67,CMP-0068,2026-04-07,2026-01-21


In [26]:
# Customers: signup dates after the data period ends, and one email on several customer ids
print("signup after 2026-10-31:", (pd.to_datetime(customers["signup_date"]) > "2026-10-31").sum())
emails = customers.dropna(subset=["email_hash"])
print("extra customer records sharing an email_hash:", emails["email_hash"].duplicated().sum())

signup after 2026-10-31: 304


extra customer records sharing an email_hash: 357


In [27]:
# fx: compare with the months we expect (Jan 2025 - Oct 2026)
expected = pd.date_range("2025-01-01", "2026-10-01", freq="MS")
missing = expected[~expected.isin(pd.to_datetime(fx["month"]))]
print("expected months:", len(expected), "| in file:", len(fx), "| missing:", list(missing.strftime("%Y-%m")))

expected months: 22 | in file: 21 | missing: ['2026-02']


## 5. Audit summary (all counts computed in code)

In [28]:
STD_DEVICE = ["Mobile", "Desktop", "Tablet"]
iso_ads    = ads["date"].str.match(r"^\d{4}-\d{2}-\d{2}$")
iso_orders = orders["order_date"].str.match(r"^\d{4}-\d{2}-\d{2}$")
cdate_s = pd.to_datetime(campaigns["start_date"]); cdate_e = pd.to_datetime(campaigns["end_date"])

rows = [
 ("fx_rates_monthly",     "month",               "Month missing (21 of 22)",                      len(missing)),
 ("campaigns",            "channel",             "Not one of the 6 official channels",             (~campaigns["channel"].isin(["Paid Search","Paid Social","Shopping","Display","Video","Email"])).sum()),
 ("campaigns",            "platform",            "Not one of the 8 official platforms",            (~campaigns["platform"].isin(["Google Ads","Microsoft Ads","Meta","TikTok","Pinterest","Google Display Network","YouTube","Klaviyo"])).sum()),
 ("campaigns",            "start/end_date",      "End date before start date",                     (cdate_e < cdate_s).sum()),
 ("campaigns",            "planned_budget",      "Blank",                                          campaigns["planned_budget"].isna().sum()),
 ("campaigns",            "campaign_name",       "Breaks naming convention",                       (~campaigns["campaign_name"].str.match(r"^(UK|US)_[A-Za-z]+_[A-Za-z]+_[A-Za-z]+_\d{6}$")).sum()),
 ("campaigns",            "campaign_name",       "Same name on several campaign_ids",              campaigns["campaign_name"].duplicated(keep=False).sum()),
 ("ad_performance_daily", "date",                "Non-ISO date format (DD/MM/YYYY)",                (~iso_ads).sum()),
 ("ad_performance_daily", "device",              "Blank",                                          ads["device"].isna().sum()),
 ("ad_performance_daily", "device",              "Case variants or 'Unknown'",                     (~ads["device"].dropna().isin(STD_DEVICE)).sum()),
 ("ad_performance_daily", "currency",            "Not GBP or USD",                                 (~ads["currency"].isin(["GBP","USD"])).sum()),
 ("ad_performance_daily", "spend",               "Blank",                                          ads["spend"].isna().sum()),
 ("ad_performance_daily", "spend",               "Negative",                                       (ads["spend"] < 0).sum()),
 ("ad_performance_daily", "spend",               "Above 2,000 (typical rows are under 100)",       (ads["spend"] > 2000).sum()),
 ("ad_performance_daily", "clicks/impressions",  "Clicks greater than impressions",                (ads["clicks"] > ads["impressions"]).sum()),
 ("ad_performance_daily", "campaign_id",         "Wrong format (lowercase / spaces)",              bad_format.sum()),
 ("ad_performance_daily", "campaign_id",         "Not found in campaigns (after trimming)",        orphans.sum()),
 ("ad_performance_daily", "conversion_value",    "Blank",                                          ads["conversion_value"].isna().sum()),
 ("ad_performance_daily", "(whole row)",         "Exact duplicate rows",                           ads.duplicated().sum()),
 ("orders",               "order_date",          "Non-ISO date format (DD/MM/YYYY)",                (~iso_orders).sum()),
 ("orders",               "country",             "Not UK or US",                                   (~orders["country"].isin(["UK","US"])).sum()),
 ("orders",               "is_refunded",         "Not Y or N (8 different encodings)",             (~orders["is_refunded"].isin(["Y","N"])).sum()),
 ("orders",               "order_value",         "Zero or negative",                               (orders["order_value"] <= 0).sum()),
 ("orders",               "order_id",            "Duplicate order_id",                             orders["order_id"].duplicated().sum()),
 ("orders",               "customer_id",         "Blank",                                          orders["customer_id"].isna().sum()),
 ("orders",               "shipping_cost",       "Blank",                                          orders["shipping_cost"].isna().sum()),
 ("orders",               "last_touch_campaign_id","Blank (legitimate: direct / unattributed)",    orders["last_touch_campaign_id"].isna().sum()),
 ("customers",            "country",             "Not UK or US",                                   (~customers["country"].isin(["UK","US"])).sum()),
 ("customers",            "crm_acquisition_channel","Blank",                                       customers["crm_acquisition_channel"].isna().sum()),
 ("customers",            "signup_date",         "After end of data period",                       (pd.to_datetime(customers["signup_date"]) > "2026-10-31").sum()),
 ("customers",            "email_hash",          "Extra records sharing an email_hash",            emails["email_hash"].duplicated().sum()),
 ("customers",            "email_hash",          "Blank",                                          customers["email_hash"].isna().sum()),
]
audit_summary = pd.DataFrame(rows, columns=["table", "column", "issue", "rows_affected"])
audit_summary["rows_affected"] = audit_summary["rows_affected"].astype(int)
audit_summary

,table,column,issue,rows_affected
0,fx_rates_monthly,month,Month missing (21 of 22),1
1,campaigns,channel,Not one of the 6 official channels,16
2,campaigns,platform,Not one of the 8 official platforms,16
3,campaigns,start/end_date,End date before start date,3
4,campaigns,planned_budget,Blank,6
5,campaigns,campaign_name,Breaks naming convention,8
6,campaigns,campaign_name,Same name on several campaign_ids,14
7,ad_performance_daily,date,Non-ISO date format (DD/MM/YYYY),1473
8,ad_performance_daily,device,Blank,1096
9,ad_performance_daily,device,Case variants or 'Unknown',4898


## 6. What the audit tells us

* **Formats are inconsistent** (dates, device, currency, country, refund flags, channel and platform labels). These are fixable with mappings.
* **Some values are impossible** (clicks above impressions, end dates before start dates, signup dates in the future) and cannot be repaired from the data alone, so they are flagged.
* **Duplicates and orphans** would distort totals: duplicate ad rows double-count spend, duplicate orders double-count revenue, and duplicate customers inflate new-customer counts.
* **Some blanks are information, not errors**: a blank last-touch campaign means the order was direct or unattributed.
* Every decision about these issues is made and recorded in `01_data_cleaning.ipynb`.